# DINOv3 Official Linear Segmentation Head In Lumen

This notebook demonstrates a full smoke-test workflow inside Lumen:
- load a downloaded DINOv3 checkpoint as the frozen encoder
- automatically pull a reproducible segmentation demo dataset from HyperData Hub when needed
- train the Lumen-native copy of the official DINOv3 linear segmentation head
- run segmentation inference with the trained head

The supervision here is intentionally rough: the HyperData dataset stores binary masks derived from the colored `reference/fete/*_anno.png` overlays so the full training path can run with minimal local setup.


## 1. Setup

If your environment is missing dependencies, run the following from the repository root:

```bash
uv sync
uv pip install -e ".[dev]"
pip install "git+https://github.com/hyper-instrument/hyper-data.git#subdirectory=client"
pip install sqlalchemy minio "pylance>=0.20.0"
uv pip install transformers matplotlib pillow
```

If the local demo dataset is missing, the notebook will try to pull it from HyperData Hub automatically via `hd hub pull`.


In [ ]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from hyperdata import HyperData
from torch.utils.data import DataLoader

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError(
        "Could not locate the Lumen repository root. "
        "Please set REPO_ROOT manually."
    )

try:
    NB_PATH = Path(__file__).resolve()
    REPO_ROOT = find_repo_root(NB_PATH)
except NameError:
    REPO_ROOT = find_repo_root(Path.cwd())

sys.path.insert(0, str(REPO_ROOT / "src"))

from lumen.data.hyperdata import HYPERDATA_AVAILABLE, HyperDataSegmentationDataset
from lumen.models import DINOv3Encoder
from lumen.training.downstream import SegmentationTrainer

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR = REPO_ROOT / "checkpoints" / "dinov3-vitl16-pretrain-lvd1689m"
HYPERDATA_DIR = REPO_ROOT / "data" / "hyperdata" / "segmentation-demo"
HUB_IDENTIFIER = os.environ.get("LUMEN_SEGMENTATION_HYPERDATA_IDENTIFIER", "@lumen/segmentation-demo")
HD_BACKEND = os.environ.get("LUMEN_HYPERDATA_BACKEND")
os.environ.setdefault("HYPERDATA_TASKS_DB", str(REPO_ROOT / "artifacts" / "hyperdata_tasks.db"))
TARGET_SIZE = 512

def run_hd_hub_pull(identifier: str, output_dir: Path, *, backend: str | None = None) -> None:
    cmd = [sys.executable, "-m", "hyperdata._core_cli", "hub", "pull", identifier, "-o", str(output_dir)]
    if backend:
        cmd.extend(["--backend", backend])
    result = subprocess.run(cmd, cwd=REPO_ROOT, env=os.environ.copy(), capture_output=True, text=True, check=False)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    if result.returncode != 0:
        raise RuntimeError(f"hd hub pull failed for {identifier} with exit code {result.returncode}")

def materialize_pulled_dataset(identifier: str, staging_dir: Path, dataset_dir: Path) -> None:
    pulled_root = staging_dir / identifier
    if not pulled_root.exists():
        raise RuntimeError(f"hd hub pull finished, but dataset root is missing: {pulled_root}")
    if dataset_dir.exists():
        shutil.rmtree(dataset_dir)
    dataset_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(pulled_root), str(dataset_dir))
    if staging_dir.exists():
        shutil.rmtree(staging_dir)

def ensure_local_hyperdata_dataset(dataset_dir: Path, identifier: str, required_keys: set[str], *, backend: str | None = None) -> tuple[HyperData, list[str]]:
    dataset_dir.parent.mkdir(parents=True, exist_ok=True)
    needs_pull = True
    if dataset_dir.exists():
        probe = HyperData(str(dataset_dir))
        needs_pull = not list(probe.keys())
    if needs_pull:
        print(f"Local HyperData dataset missing or empty. Pulling from Hub: {identifier}")
        staging_dir = dataset_dir.parent / f".{dataset_dir.name}_pull"
        if staging_dir.exists():
            shutil.rmtree(staging_dir)
        run_hd_hub_pull(identifier, staging_dir, backend=backend)
        materialize_pulled_dataset(identifier, staging_dir, dataset_dir)
    dataset = HyperData(str(dataset_dir))
    dataset_keys = list(dataset.keys())
    missing_keys = required_keys.difference(dataset_keys)
    if missing_keys:
        raise RuntimeError(f"HyperData dataset at {dataset_dir} is missing keys: {sorted(missing_keys)}")
    return dataset, dataset_keys

if not CHECKPOINT_DIR.exists():
    raise FileNotFoundError(f"Missing DINOv3 checkpoint directory: {CHECKPOINT_DIR}")
if not HYPERDATA_AVAILABLE:
    raise ImportError("HyperData is not installed. Install the thin client with: pip install 'git+https://github.com/hyper-instrument/hyper-data.git#subdirectory=client' and pip install sqlalchemy minio 'pylance>=0.20.0'")

hd, dataset_keys = ensure_local_hyperdata_dataset(HYPERDATA_DIR, HUB_IDENTIFIER, {"images", "masks", "image_ids"}, backend=HD_BACKEND)
image_ids_raw = hd["image_ids"]
image_ids = image_ids_raw.to_numpy() if hasattr(image_ids_raw, "to_numpy") else np.asarray(image_ids_raw)

print(f"Repo root: {REPO_ROOT}")
print(f"Device: {DEVICE}")
print(f"Checkpoint: {CHECKPOINT_DIR}")
print(f"HyperData dataset: {HYPERDATA_DIR}")
print(f"Hub identifier: {HUB_IDENTIFIER}")
print(f"Dataset keys: {dataset_keys}")
print(f"Training samples: {[str(item) for item in image_ids.tolist()]}")


## 2. Load The HyperData Smoke-Test Dataset

This notebook uses a local HyperData cache at `data/hyperdata/segmentation-demo`. If the cache is missing, the setup cell automatically runs `hd hub pull` against the configured Hub identifier.

The goal is not label quality. The goal is to make the complete DINOv3 -> train head -> inference path runnable in one notebook without depending on a private local data layout.


In [ ]:
dataset = HyperDataSegmentationDataset(
    hd,
    image_array="images",
    mask_array="masks",
    image_size=TARGET_SIZE,
    channels=1,
)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

sample = dataset[0]
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(sample["image"].squeeze(0).numpy(), cmap="gray")
axes[0].set_title(f"Training Image: {image_ids[0]}")
axes[0].axis("off")
axes[1].imshow(sample["mask"].numpy(), cmap="gray")
axes[1].set_title("Coarse Smoke-Test Mask")
axes[1].axis("off")
plt.tight_layout()

print(f"Dataset size: {len(dataset)}")
print(f"Batch image shape: {next(iter(loader))['image'].shape}")
print(f"Stored image array shape: {hd['images'].shape}")
print(f"Stored mask array shape: {hd['masks'].shape}")


## 3. Build DINOv3 + Official Linear Head Through Lumen

The encoder is frozen (`trainability="head_only"`) and only the official DINOv3 linear segmentation head is trained.


In [ ]:
encoder = DINOv3Encoder(
    model_dir=CHECKPOINT_DIR,
    device=DEVICE,
    local_files_only=True,
)
encoder.eval()

trainer = SegmentationTrainer(
    encoder=encoder,
    num_classes=2,
    trainability="head_only",
    segmentation_head_name="dinov3-linear",
    segmentation_loss="ce",
    scheduler_name="none",
    lr=1e-3,
).to(DEVICE)

trainable_params = sum(p.numel() for p in trainer.parameters() if p.requires_grad)
print(f"Encoder patch size: {encoder.patch_size}")
print(f"Encoder embed dim: {encoder.embed_dim}")
print(f"Trainable params: {trainable_params:,}")
print(f"Head class: {type(trainer.head).__name__}")


## 4. Train The Head

This is only a smoke-test training loop. A few epochs are enough to verify that the Lumen trainer can now run the official DINOv3 linear head end to end.


In [ ]:
EPOCHS = 3
loss_history: list[float] = []
trainer.train()

for epoch in range(EPOCHS):
    epoch_losses = []
    for batch in loader:
        batch = {key: value.to(DEVICE) for key, value in batch.items()}
        metrics = trainer.train_step(batch)
        epoch_losses.append(metrics["loss"])
    mean_loss = float(np.mean(epoch_losses))
    loss_history.append(mean_loss)
    print(f"Epoch {epoch + 1}/{EPOCHS} - loss: {mean_loss:.4f}")

plt.figure(figsize=(6, 3))
plt.plot(range(1, EPOCHS + 1), loss_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Head Training Smoke Test")
plt.grid(True, alpha=0.3)
plt.tight_layout()


## 5. Run Segmentation Inference

After the smoke-test training loop, run the trained head on a demo image and visualize the foreground probability map.


In [ ]:
demo_index = 0
demo_sample = dataset[demo_index]
demo_tensor = demo_sample["image"].unsqueeze(0).to(DEVICE)
demo_image_id = str(image_ids[demo_index])
input_image = demo_sample["image"].squeeze(0).cpu().numpy()

trainer.eval()
with torch.inference_mode():
    logits = trainer(demo_tensor)
    probs = torch.softmax(logits, dim=1).cpu()
    pred_mask = probs.argmax(dim=1)[0]

print(f"Logits shape: {tuple(logits.shape)}")
print(f"Probability shape: {tuple(probs.shape)}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(input_image, cmap="gray")
axes[0].set_title(f"Input Image: {demo_image_id}")
axes[0].axis("off")

axes[1].imshow(probs[0, 1].numpy(), cmap="magma")
axes[1].set_title("Foreground Probability")
axes[1].axis("off")

axes[2].imshow(pred_mask.numpy(), cmap="gray")
axes[2].set_title("Predicted Mask")
axes[2].axis("off")
plt.tight_layout()


## 6. Save And Reload Only The Trained Head

Because the backbone already comes from a separate DINOv3 checkpoint, experiments can save only the segmentation head weights.


In [ ]:
HEAD_WEIGHTS = REPO_ROOT / "artifacts" / "dinov3_official_linear_head_demo.pt"
HEAD_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)

torch.save(trainer.head.state_dict(), HEAD_WEIGHTS)
print(f"Saved head weights to: {HEAD_WEIGHTS}")

state = torch.load(HEAD_WEIGHTS, map_location="cpu", weights_only=True)
trainer.head.load_state_dict(state)
print("Reloaded head weights successfully.")


## 7. Notes

This notebook is a runnable baseline example, not a production training recipe.
- the encoder is frozen and only the official linear head is optimized
- the masks come from coarse colored overlay extraction, so they are only for smoke testing
- the dataset now comes from a local HyperData package, which makes the demo easier to copy and share
- for real experiments, replace the tiny demo dataset with proper segmentation masks or pseudo masks
